In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import glob

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("Dataset: ")
for file in csv_files:
    print(file)

file_path = csv_files[0]

# Đọc 10.000 dòng
df = pd.read_csv(file_path, nrows=10000)

# Xóa khoảng trắng thừa trong tên cột nếu có
df.columns = df.columns.str.strip()
df = df.drop(columns=["Unnamed: 0"])

print("\nĐọc dữ liệu thành công!")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])

df.head()

# sensor_columns = [
#     "TP2",
#     "TP3",
#     "Oil_temperature",
#     "Motor_current"
# ]

# df[sensor_columns].describe().round(2)


# Các giá trị đo được từ cảm biến của máy nén khi

| Cột               | Ý nghĩa đơn giản                       |
| ----------------- | -------------------------------------- |
| `timestamp`       | Thời điểm ghi dữ liệu                  |
| `TP2`             | Áp suất khí tại đầu ra máy nén         |
| `TP3`             | Áp suất khí trong hệ thống             |
| `H1`              | Áp suất sau bộ sấy khí                 |
| `DV_pressure`     | Áp suất liên quan đến quá trình xả khí |
| `Reservoirs`      | Áp suất trong bình chứa khí            |
| `Oil_temperature` | Nhiệt độ dầu máy nén                   |
| `Motor_current`   | Dòng điện động cơ                      |


# Các trạng thái của máy nén khí

| Cột               | Ý nghĩa đơn giản            |
| ----------------- | --------------------------- |
| `COMP`            | Trạng thái máy nén          |
| `DV_eletric`      | Trạng thái van xả điện      |
| `Towers`          | Trạng thái hai tháp sấy khí |
| `MPG`             | Tín hiệu điều khiển máy nén |
| `LPS`             | Tín hiệu áp suất thấp       |
| `Pressure_switch` | Trạng thái công tắc áp suất |
| `Oil_level`       | Trạng thái mức dầu          |
| `Caudal_impulses` | Xung đo lưu lượng khí       |


# Ý nghĩa các giá trị tại 1 record

| Giá trị                  | Ý nghĩa chính xác                                                             |
| ------------------------------- | ----------------------------------------------------------------------------- |
| `timestamp = 2/1/2020 12:00 AM` | Ngày 1/2/2020, lúc 00:00                                                      |
| `TP2 = -0.012 bar`              | Áp suất tại máy nén gần bằng 0                                                |
| `TP3 = 9.358 bar`               | Áp suất tại bảng khí nén là 9,358 bar                                         |
| `H1 = 9.340 bar`                | Áp suất liên quan đến lúc bộ lọc tách nước xả                                 |
| `DV_pressure = -0.024 bar`      | Gần bằng 0. Theo tài liệu, giá trị 0 thường xuất hiện khi máy nén chạy có tải |
| `Reservoirs = 9.358 bar`        | Áp suất phía sau bình chứa; gần bằng `TP3` là đúng với mô tả                  |
| `Oil_temperature = 53.6°C`      | Nhiệt độ dầu trong máy nén                                                    |
| `Motor_current = 0.04 A`        | Gần 0 A → động cơ đang tắt                                                    |
| `COMP = 1`                      | Van hút khí đang ở trạng thái không hút khí → máy tắt hoặc chạy không tải     |
| `DV_eletric = 0`                | Van đầu ra không hoạt động → máy tắt hoặc chạy không tải                      |
| `Towers = 1`                    | Tháp sấy khí số 2 đang hoạt động                                              |
| `MPG = 1`                       | Tín hiệu điều khiển khởi động máy nén đang được kích hoạt                     |
| `LPS = 0`                       | Không có tín hiệu áp suất thấp dưới 7 bar                                     |
| `Pressure_switch = 1`           | Phát hiện hoạt động xả tại tháp sấy khí                                       |
| `Oil_level = 1`                 | Tín hiệu mức dầu thấp đang được kích hoạt                                     |
| `Caudal_impulses = 1`           | Ghi nhận xung của lượng khí đi từ APU đến bình chứa                           |


**Lưu ý: Số âm rất nhỏ do cảm biến/van do nhiễu điện từ nên không thể đúng 100%, các giá trị này sẽ được hiệu chuẩn trong thiết bị, ko cần quan tâm sai số nhỏ này.*

**> Kết luận: Tại thời điểm này, động cơ của máy nén khí đang tắt hoặc ở trạng thái không tải vì không tiêu thụ điện, bình chứa vẫn có ~9.36 bar áp suất**


In [ ]:
import matplotlib.pyplot as plt

# Lấy 1.000 record đầu tiên
sample = df.iloc[:5000]

# Vì dữ liệu ghi mỗi giây một lần
seconds = range(len(sample))

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Áp suất tại máy nén
axes[0].plot(seconds, sample["TP2"], color="blue")
axes[0].set_ylabel("TP2 (bar)")
axes[0].set_title("Áp suất tại máy nén")

# Áp suất của hệ thống
axes[1].plot(seconds, sample["TP3"], color="orange")
axes[1].set_ylabel("TP3 (bar)")
axes[1].set_title("Áp suất tại bảng khí nén")

# Dòng điện động cơ
axes[2].plot(seconds, sample["Motor_current"], color="green")
axes[2].set_ylabel("Motor_current (A)")
axes[2].set_xlabel("Thời gian (giây)")
axes[2].set_title("Dòng điện động cơ")

plt.tight_layout()
plt.show()

In [ ]:
# Đọc toàn bộ dataset

import pandas as pd


df = pd.read_csv(file_path)

# Xóa cột số thứ tự thừa
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Chuyển timestamp thành kiểu thời gian
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="%Y-%m-%d %H:%M:%S"
)

print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("Thời điểm đầu:", df["timestamp"].min())
print("Thời điểm cuối:", df["timestamp"].max())

In [ ]:
import pandas as pd

# Khoảng thời gian lỗi từ báo cáo bảo trì
fault_start = pd.Timestamp("2020-04-18 23:30:00")
fault_end = pd.Timestamp("2020-04-18 23:59:59")

# Lọc các record nằm trong khoảng lỗi
fault_1 = df[
    (df["timestamp"] >= fault_start) &
    (df["timestamp"] <= fault_end)
].copy()

print("Số record lấy được:", len(fault_1))

# Xem 5 record đầu
display(fault_1.head())

# Xem 5 record cuối
display(fault_1.tail())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)

# Áp suất tại máy nén
axes[0].plot(
    fault_1["timestamp"],
    fault_1["TP2"],
    color="blue",
    linewidth=0.7
)
axes[0].set_ylabel("TP2 (bar)")
axes[0].set_title("Áp suất tại máy nén")

# Áp suất tại bảng khí nén
axes[1].plot(
    fault_1["timestamp"],
    fault_1["TP3"],
    color="orange",
    linewidth=0.7
)
axes[1].set_ylabel("TP3 (bar)")
axes[1].set_title("Áp suất tại bảng khí nén")

# Dòng điện động cơ
axes[2].plot(
    fault_1["timestamp"],
    fault_1["Motor_current"],
    color="green",
    linewidth=0.7
)
axes[2].set_ylabel("Dòng điện (A)")
axes[2].set_xlabel("Thời gian")
axes[2].set_title("Dòng điện động cơ")

fig.suptitle(
    "Dữ liệu trong khoảng được báo cáo rò khí: 18/04/2020",
    fontsize=14
)

plt.tight_layout()
plt.show()

In [ ]:
# Ban đầu coi tất cả record là bình thường
df["label"] = 0

# Xác định khoảng rò khí lần đầu
fault_start = pd.Timestamp("2020-04-18 00:00:00")
fault_end = pd.Timestamp("2020-04-18 23:59:59")

# Gắn nhãn 1 cho các record trong khoảng đó
df.loc[
    (df["timestamp"] >= fault_start) &
    (df["timestamp"] <= fault_end),
    "label"
] = 1

# Đếm số lượng từng nhãn
print(df["label"].value_counts())

fault_rows = df[df["label"] == 1]

print("Tổng record:", len(fault_rows))
print("Số timestamp khác nhau:", fault_rows["timestamp"].nunique())
print(
    "Số timestamp bị lặp:",
    len(fault_rows) - fault_rows["timestamp"].nunique()
)

In [ ]:
# Mốc chia dữ liệu
split_time = pd.Timestamp("2020-03-01 00:00:00")

# Tháng 2: dữ liệu huấn luyện
train_df = df[df["timestamp"] < split_time]

# Từ tháng 3 trở đi: dữ liệu kiểm tra
test_df = df[df["timestamp"] >= split_time]

print("Số record huấn luyện:", len(train_df))
print("Số record kiểm tra:", len(test_df))

print("\nNhãn trong dữ liệu huấn luyện:")
print(train_df["label"].value_counts())

print("\nNhãn trong dữ liệu kiểm tra:")
print(test_df["label"].value_counts())